# Instruction Fine-Tuning Notebook
## Stage 2: Q&A Training

This notebook performs instruction fine-tuning on question-answer pairs to teach the model how to respond to user queries.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from datasets import Dataset
from peft import LoraConfig, get_peft_model
import json
import os

print("=" * 60)
print("STAGE 2: INSTRUCTION FINE-TUNING (SFT)")
print("=" * 60)

print(f\"GPU Available: {torch.cuda.is_available()}\")
if torch.cuda.is_available():
    print(f\"GPU Name: {torch.cuda.get_device_name(0)}\\n")

## Step 1: Install Libraries

In [ ]:
!pip install -q torch transformers datasets peft bitsandbytes accelerate unsloth[colab-new] -U

## Step 2: Load Instruction Dataset

In [ ]:
print("[STEP 1] Loading instruction dataset...")
dataset_path = 'course-doubt-assistant/data/instruction_dataset.jsonl'

data = []
with open(dataset_path, 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line))

print(f\"✓ Loaded {len(data)} instruction-response pairs\")
print(f\"\\nFirst example:\")
print(f\"  Instruction: {data[0]['instruction']}\")
print(f\"  Response: {data[0]['response'][:100]}...\\n")

## Step 3: Format Instruction Dataset

In [ ]:
print("[STEP 2] Formatting dataset...")

def format_instruction(example):
    return {
        'text': f\"### Instruction:
{example['instruction']}
### Response:
{example['response']}"
    }

formatted_data = [format_instruction(d) for d in data]
print("✓ First formatted example:")
print(formatted_data[0]['text'][:200])

## Step 4: Create Dataset Object

In [ ]:
print("\n[STEP 3] Creating dataset object...")
dataset = Dataset.from_dict({'text': [d['text'] for d in formatted_data]})
print(f\"✓ Dataset size: {len(dataset)}\")
print(f\"✓ Dataset columns: {dataset.column_names}\\n")

## Step 5: Load Model and Tokenizer

In [ ]:
print("[STEP 4] Loading model and tokenizer...")
MODEL_NAME = 'unsloth/tinyllama-bnb-4bit'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map='auto',
    torch_dtype=torch.float16
)

print(f\"✓ Model loaded: {MODEL_NAME}\\n")

## Step 6: Configure LoRA for Instruction Tuning

In [ ]:
print("[STEP 5] Configuring LoRA...")
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'v_proj'],
    modules_to_save=['lm_head']
)

model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f\"✓ Trainable params: {trainable_params:,}\")
print(f\"✓ Total params: {total_params:,}\")
print(f\"✓ Trainable %: {100 * trainable_params / total_params:.2f}%\\n")

## Step 7: Tokenize Dataset

In [ ]:
print("[STEP 6] Tokenizing dataset...")

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=512
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['text']
)

print(f\"✓ Tokenized dataset size: {len(tokenized_dataset)}\\n")

## Step 8: Configure Training Arguments

In [ ]:
print("[STEP 7] Configuring training arguments...")
os.makedirs('./outputs/instruction_ft', exist_ok=True)

training_args = TrainingArguments(
    output_dir='./outputs/instruction_ft',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=1e-4,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=5,
    save_steps=25,
    save_total_limit=2,
    gradient_accumulation_steps=4,
    optim='adamw_8bit',
    seed=42,
    report_to=[]
)

print("✓ Training arguments configured\\n")

## Step 9: Train Model

In [ ]:
print("[STEP 8] Starting instruction fine-tuning...")
print("-" * 60)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
)

train_result = trainer.train()
print("-" * 60)
print(f\"✓ Training loss: {train_result.training_loss:.4f}\\n")

## Step 10: Save Model

In [ ]:
print("[STEP 9] Saving SFT adapter...")
os.makedirs('./models/sft_adapter', exist_ok=True)

sft_adapter_path = './models/sft_adapter'
model.save_pretrained(sft_adapter_path)
tokenizer.save_pretrained(sft_adapter_path)

print(f\"✓ SFT adapter saved to {sft_adapter_path}\\n")

## Step 11: Test SFT Model

In [ ]:
print("[STEP 10] Testing SFT fine-tuned model...")
print("-" * 60)

def generate_response(instruction, max_length=150):
    prompt = f\"### Instruction:
{instruction}
### Response:
\"
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    outputs = model.generate(
        inputs.input_ids,
        max_length=max_length,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

test_questions = [
    'What is machine learning?',
    'Explain gradient descent',
    'What is a neural network?'
]

for q in test_questions:
    print(f\"\\nQuestion: {q}\")
    answer = generate_response(q)
    print(f\"Answer: {answer[:150]}...\")

print("\\n" + "=" * 60)
print("✓ STAGE 2 COMPLETE!")
print("=" * 60)